# A tiny GPT, built by hand

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jland123/simplewiki-gpt/blob/main/simplewiki_gpt.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/jland123/simplewiki-gpt/blob/main/simplewiki_gpt.ipynb)

This notebook builds a small **language model** from scratch and trains it on [Simple English Wikipedia](https://simple.wikipedia.org).
A language model is a program that reads some text and guesses what comes next. Do that over and over (guess a word, add it to the text, guess the next one) and it writes.

The design is the same one used by GPT, the family of models behind ChatGPT, just far smaller.
Every layer is written out by hand as a few lines of math instead of using PyTorch's ready-made building blocks, so nothing is hidden.

**Steps**
1. Settings
2. Load the text
3. Turn the text into numbers
4. Build the model
5. Train it
6. Check how well it learned
7. Let it write

**Before you start, turn on a GPU** (a graphics card, which does this kind of math much faster than a normal processor):
- **Kaggle:** right-hand panel → *Session options* → *Accelerator* → a GPU option.
- **Colab:** *Runtime* → *Change runtime type* → *T4 GPU*.

Without a GPU the notebook still runs, but training with the full settings is far too slow.

## 1. Settings

Every number you might want to change lives in this one cell.

- Getting a **"CUDA out of memory"** error? (CUDA is the software that runs code on Nvidia GPUs; the error means the GPU ran out of room.) Lower `BATCH_SIZE`, for example to 32, or lower `DIMS`.
- Just want a quick test run? Set `MAX_LINES = 20_000` and `EPOCHS = 1`.

In [ ]:
import os
# Lets PyTorch reuse GPU memory more efficiently. Must be set before PyTorch first touches the GPU.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch

# --- Data ---
DATA_PATH = None          # None = find or download Simple English Wikipedia automatically.
                          # Or set it to any plain .txt file with one paragraph per line.
MAX_LINES = 250_000       # how many non-empty lines (paragraphs) to read
CHUNK_CHARS = 500         # long paragraphs are cut into pieces of this many characters
MIN_CHUNK_CHARS = 50      # pieces shorter than this are thrown away
MAX_TOKENS = 169          # each training example is cut to at most this many tokens
MIN_TOKEN_COUNT = 50      # a token must appear at least this many times to get its own ID

# --- Model ---
DIMS = 512                # how many numbers describe each token inside the model
N_LAYERS = 4              # how many transformer blocks are stacked
N_HEADS = 4               # attention heads per block (DIMS must divide evenly by this)

# --- Training ---
EPOCHS = 5                # full passes over the training data
BATCH_SIZE = 50           # examples processed at once; lower it if the GPU runs out of memory
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
SEED = 42                 # fixes the random choices so runs are repeatable
CHECKPOINT_PATH = "weights.pt"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("using", device)
if device == "cpu":
    print("No GPU found - training will be very slow. See 'Before you start' above.")

## 2. Load the text

The dataset is a plain-text copy of Simple English Wikipedia, a version of Wikipedia written with shorter sentences and easier words. Each line of the file is one paragraph. We read the first `MAX_LINES` non-empty lines.

- **On Kaggle:** attach the dataset first. Click *Add Input*, search for **plain text wikipedia simpleenglish** (by *ffatty*), and add it.
- **Anywhere else:** this cell downloads it for you with the `kagglehub` library. It may ask you to sign in to Kaggle.

In [ ]:
FILENAME = "simplewiki_01_of_01.txt"

def find_file(folder, name):
    for root, _, files in os.walk(folder):
        if name in files:
            return os.path.join(root, name)
    return None

def find_data_file():
    # On Kaggle, attached datasets live under /kaggle/input.
    path = find_file("/kaggle/input", FILENAME)
    if path:
        return path
    # Anywhere else, download the dataset (needs `pip install kagglehub`).
    import kagglehub
    folder = kagglehub.dataset_download("ffatty/plain-text-wikipedia-simpleenglish")
    return find_file(folder, FILENAME)

path = DATA_PATH or find_data_file()
print("reading", path)

paragraphs = []
with open(path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            paragraphs.append(line)
            if len(paragraphs) == MAX_LINES:
                break

print(f"{len(paragraphs):,} paragraphs loaded; the longest is {max(map(len, paragraphs)):,} characters")
print("example:", paragraphs[1][:300])

## 3. Turn the text into numbers

A neural network only works with numbers, so the text has to be converted first.

**Tokens.** We split text into *tokens*: common chunks of text such as whole words (`" river"`), pieces of words (`"ing"`), or punctuation (`","`). We use the same tokenizer as GPT-2, from the `tiktoken` library. A *tokenizer* is the tool that does this splitting, and it gives every token an ID number.

**Our own vocabulary.** The *vocabulary* is the list of every token the model knows. GPT-2's has 50,257 tokens, but many of them never show up in Simple Wikipedia. Each token the model knows costs memory, so we keep only tokens that appear at least `MIN_TOKEN_COUNT` times and renumber them 1, 2, 3, and so on. All the rare ones share a single "unknown" ID, shown as `<?>`.

Before splitting into tokens we also:
- **lowercase** everything, so "River" and "river" become the same token, and
- **cut long paragraphs** into pieces of `CHUNK_CHARS` characters, so no single example is huge.

In [ ]:
import tiktoken
from collections import Counter

enc = tiktoken.get_encoding("gpt2")

# 1. Lowercase, then cut every paragraph into pieces of CHUNK_CHARS characters.
chunks = []
for p in paragraphs:
    p = p.lower()
    for i in range(0, len(p), CHUNK_CHARS):
        piece = p[i:i + CHUNK_CHARS]
        if len(piece) >= MIN_CHUNK_CHARS:
            chunks.append(piece)

# 2. Split each piece into GPT-2 tokens, keeping at most MAX_TOKENS of them.
seqs = [enc.encode_ordinary(c)[:MAX_TOKENS] for c in chunks]

# 3. Build our own smaller vocabulary from the tokens that appear often enough.
counts = Counter(t for s in seqs for t in s)
used = sorted(t for t, c in counts.items() if c >= MIN_TOKEN_COUNT)   # the GPT-2 IDs we keep

BOUNDARY = 0                                   # marks the start and the end of every piece
UNK = len(used) + 1                            # stands in for every rare token we dropped
g2l = {g: i + 1 for i, g in enumerate(used)}   # GPT-2 ID -> our ID
vocab_size = len(used) + 2

itos = {i + 1: enc.decode([g]) for i, g in enumerate(used)}   # our ID -> readable text
itos[BOUNDARY], itos[UNK] = "<s>", "<?>"

rare = sum(c for c in counts.values() if c < MIN_TOKEN_COUNT) / sum(counts.values())
print(f"{len(chunks):,} pieces of text, {sum(map(len, seqs)) / len(seqs):.1f} tokens each on average")
print(f"vocabulary: {vocab_size:,} tokens (GPT-2 has {enc.n_vocab:,}); {rare:.2%} of all tokens are rare and become <?>")
print("a piece split into tokens:", [enc.decode([t]) for t in seqs[0][:15]])

### Inputs and targets

The model's whole job is: *given the tokens so far, guess the next one.* So from each piece of text we make two rows of IDs:

- the **input**: a start marker (ID 0, shown as `<s>`) followed by the piece's tokens, and
- the **target**: the same row shifted one step to the left, with the marker at the end.

At every position, the target holds the token the model should guess after reading the input up to that point.

Pieces have different lengths, but the GPU wants one neat rectangle of numbers (a *tensor*). So shorter rows are padded at the end. Padded spots in the target get the value `-100`, which tells the loss function (explained in step 5) to ignore them.

In [ ]:
block_size = max(len(s) for s in seqs) + 1    # longest piece, plus the start marker

N = len(seqs)
inputs = torch.zeros((N, block_size), dtype=torch.long)          # padding = 0
targets = torch.full((N, block_size), -100, dtype=torch.long)    # -100 = ignore this spot
for row, s in enumerate(seqs):
    ids = [BOUNDARY] + [g2l.get(t, UNK) for t in s] + [BOUNDARY]
    n = len(ids) - 1
    inputs[row, :n] = torch.tensor(ids[:-1])
    targets[row, :n] = torch.tensor(ids[1:])

print("input :", "|".join(itos[t] for t in inputs[0, :12].tolist()))
print("target:", "|".join(itos.get(t, "_") for t in targets[0, :12].tolist()))   # _ = padding

### Training and validation data

We shuffle the examples and set aside 10% as **validation** data. The model never trains on these. We only use them to check whether it learned general patterns of English or just memorized its training text.

The data stays in normal memory, and each batch is copied to the GPU only when it is needed. That leaves more GPU memory for the model.

In [ ]:
g = torch.Generator().manual_seed(SEED)   # same split every run
perm = torch.randperm(N, generator=g)
inputs, targets = inputs[perm], targets[perm]

n_train = int(0.9 * N)
train_x, train_y = inputs[:n_train], targets[:n_train]
val_x, val_y = inputs[n_train:], targets[n_train:]
print(f"training examples: {len(train_x):,}   validation examples: {len(val_x):,}")

## 4. Build the model

**The big idea.** Inside the model, each token is a list of `DIMS` numbers, called a *vector*. Think of it as the model's current notes about what that token means here. The tokens' vectors pass through a stack of layers, and each layer updates every vector a little. At the end, each token's vector is turned into a score for every token in the vocabulary, answering: "how likely is this to come next?"

The pieces:

- **Embeddings** are lookup tables that give each token its starting vector. The *token embedding* has one row per token ID. The *position embedding* has one row per position (1st, 2nd, 3rd, ...) and is added on, so the model knows the order of words. Both start as small random numbers and are learned during training.
- **Attention** is how tokens share information. Each token asks a question (its *query*), every earlier token offers a label (its *key*), and the better a key matches the question, the more of that token's information (its *value*) gets passed along. A token may only look at tokens *before* it. Otherwise it could cheat by peeking at the word it is supposed to guess. One copy of this is called a **head**. Several heads side by side let the model follow several kinds of connections at once.
- **Feed-forward** is a small two-layer network that each token's vector goes through on its own, after attention. Attention gathers information; this step works with it.
- **LayerNorm** (layer normalization) rescales each vector so its numbers average 0 with a spread of 1. That stops numbers from growing huge or shrinking to nothing as they pass through many layers, which keeps training stable.
- **Residual connections**: attention and feed-forward each *add* their result onto the vector instead of replacing it. Nothing from earlier layers gets lost, and deep stacks train much better.

One attention step plus one feed-forward step make one **transformer block**. We stack `N_LAYERS` of them.

Every *parameter* (each number in these tables and layers) starts random and gets adjusted during training.
Each class below holds its own parameters and lists them in `parameters()`.

In [ ]:
class Linear:
    """out = x @ weight + bias: every output is a weighted sum of all the inputs."""
    def __init__(self, fan_in, fan_out, bias=True, gain=1.0):
        self.weight = torch.randn(fan_in, fan_out) * gain / fan_in ** 0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x):
        out = x @ self.weight
        if self.bias is not None:
            out = out + self.bias
        return out

    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])


class LayerNorm:
    """Rescale each vector to average 0 and spread 1, then apply a learned scale and shift."""
    def __init__(self, dims, eps=1e-8):
        self.shift = torch.zeros(dims)
        self.scale = torch.ones(dims)
        self.eps = eps

    def __call__(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, correction=0, keepdim=True)
        return (x - mean) / (std + self.eps) * self.scale + self.shift

    def parameters(self):
        return [self.shift, self.scale]


class AttentionHead:
    """One attention head. Takes (batch, tokens, dims) and returns (batch, tokens, head_dims)."""
    def __init__(self, dims, head_dims):
        self.head_dims = head_dims
        self.query = torch.randn(head_dims, dims) / dims ** 0.5
        self.key = torch.randn(head_dims, dims) / dims ** 0.5
        self.value = torch.randn(head_dims, dims) / dims ** 0.5

    def __call__(self, x):
        T = x.shape[1]
        q = x @ self.query.T     # what each token is looking for
        k = x @ self.key.T       # what each token has to offer
        v = x @ self.value.T     # the information each token passes along
        scores = q @ k.mT / self.head_dims ** 0.5      # (batch, T, T): how well each query matches each key
        future = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1)
        scores = scores.masked_fill(future, float("-inf"))   # no looking at later tokens
        weights = scores.softmax(dim=-1)                     # each row becomes percentages that add up to 1
        return weights @ v

    def parameters(self):
        return [self.query, self.key, self.value]


class MultiHeadAttention:
    """LayerNorm, then several attention heads side by side, then add the result back on (residual)."""
    def __init__(self, dims, n_heads):
        if dims % n_heads != 0:
            raise ValueError(f"DIMS ({dims}) must divide evenly by N_HEADS ({n_heads})")
        self.heads = [AttentionHead(dims, dims // n_heads) for _ in range(n_heads)]
        self.proj = Linear(dims, dims, bias=False)   # mixes the heads' outputs together
        self.norm = LayerNorm(dims)

    def __call__(self, x):
        h = self.norm(x)
        h = torch.cat([head(h) for head in self.heads], dim=-1)
        return x + self.proj(h)

    def parameters(self):
        return [p for head in self.heads for p in head.parameters()] + self.proj.parameters() + self.norm.parameters()


class FeedForward:
    """LayerNorm, then a two-layer network on each token by itself, then add the result back on (residual)."""
    def __init__(self, dims):
        self.norm = LayerNorm(dims)
        self.expand = Linear(dims, 4 * dims, gain=2 ** 0.5)
        self.shrink = Linear(4 * dims, dims)

    def __call__(self, x):
        return x + self.shrink(torch.relu(self.expand(self.norm(x))))   # relu: keep positives, zero out negatives

    def parameters(self):
        return self.norm.parameters() + self.expand.parameters() + self.shrink.parameters()

Now put the pieces together: embeddings, then `N_LAYERS` transformer blocks, then a final LayerNorm and a `Linear` layer that gives a score to every token in the vocabulary.

In [ ]:
torch.manual_seed(SEED)

token_embedding = torch.randn(vocab_size, DIMS) * 0.02      # one row per token ID
position_embedding = torch.randn(block_size, DIMS) * 0.02   # one row per position

layers = []
for _ in range(N_LAYERS):
    layers += [MultiHeadAttention(DIMS, N_HEADS), FeedForward(DIMS)]
layers += [LayerNorm(DIMS), Linear(DIMS, vocab_size, gain=0.1)]

params = [token_embedding] + [p for layer in layers for p in layer.parameters()] + [position_embedding]
for p in params:
    p.data = p.data.to(device)   # move in place, so the layers still point at the same tensors
    p.requires_grad_()           # ask PyTorch to work out how to adjust this during training

def forward(x):
    """(batch, tokens) of token IDs -> (batch, tokens, vocab_size) scores for the next token."""
    h = token_embedding[x] + position_embedding[:x.shape[1]]
    for layer in layers:
        h = layer(h)
    return h

print(f"the model has {sum(p.numel() for p in params):,} parameters")

## 5. Train

Training means: show the model some examples, measure how wrong its guesses were, nudge all its parameters so it would have been a little less wrong, and repeat many times.

- The **loss** measures how wrong it was. We use *cross-entropy*: at every position, look at the probability the model gave to the real next token. Confident and right gives a loss near 0. Giving the real token a tiny probability gives a big loss. We average this over every position. Lower is better.
- A **batch** is a group of `BATCH_SIZE` examples handled at once. GPUs are fast at doing many at the same time.
- An **epoch** is one full pass through the training data.
- The **optimizer** (here AdamW) decides how to nudge each parameter after every batch. The **learning rate** sets how big the nudges are. **Weight decay** gently pulls parameters toward zero, which discourages memorizing.

A useful reference point: a model that knows nothing and spreads its guesses evenly over every token gets a loss of ln(vocabulary size), the natural logarithm. The untrained model should start close to that:

In [ ]:
import math
import torch.nn.functional as F

def loss_on(x, y):
    """Average cross-entropy loss of the model on a batch of inputs x and targets y."""
    x, y = x.to(device), y.to(device)
    return F.cross_entropy(forward(x).reshape(-1, vocab_size), y.reshape(-1), ignore_index=-100)

with torch.no_grad():
    print(f"untrained model: {loss_on(train_x[:BATCH_SIZE], train_y[:BATCH_SIZE]).item():.3f}")
print(f"random guessing: {math.log(vocab_size):.3f}")

This is the long part. The progress bar shows the loss of the latest batch (`loss`) and the average so far this epoch (`avg`).
After every epoch we check the validation loss and save the parameters to `CHECKPOINT_PATH` (`weights.pt`). On Kaggle you can find that file in the *Output* panel.

Already trained earlier and saved the file? Skip this cell and run the *Load saved weights* cell below instead.

In [ ]:
from tqdm.auto import tqdm

@torch.no_grad()
def evaluate(x, y, batch_size=BATCH_SIZE):
    """Average loss per token over a whole dataset, without training."""
    total, count = 0.0, 0
    for i in range(0, len(x), batch_size):
        xb, yb = x[i:i + batch_size], y[i:i + batch_size]
        n = (yb != -100).sum().item()
        total += loss_on(xb, yb).item() * n
        count += n
    return total / count

def save_checkpoint(path=CHECKPOINT_PATH):
    torch.save({"params": [p.detach().cpu() for p in params], "used": used}, path)

opt = torch.optim.AdamW(params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
loss_history = []   # (step, loss) every 50 steps, for the chart below
step = 0

for epoch in range(EPOCHS):
    order = torch.randperm(len(train_x))      # new shuffle each epoch
    n_batches = len(train_x) // BATCH_SIZE    # drop the last partial batch; equal-size batches use GPU memory more predictably
    total = 0.0
    bar = tqdm(range(n_batches), desc=f"epoch {epoch + 1}/{EPOCHS}")
    for b in bar:
        idx = order[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
        loss = loss_on(train_x[idx], train_y[idx])
        opt.zero_grad(set_to_none=True)
        loss.backward()   # work out which direction to nudge every parameter
        opt.step()        # nudge them

        total += loss.item()
        step += 1
        if step % 50 == 0:
            loss_history.append((step, loss.item()))
            bar.set_postfix(loss=f"{loss.item():.3f}", avg=f"{total / (b + 1):.3f}")

    print(f"epoch {epoch + 1}: training loss {total / n_batches:.4f}, validation loss {evaluate(val_x, val_y):.4f}")
    save_checkpoint()

In [ ]:
import matplotlib.pyplot as plt

if loss_history:
    steps, losses = zip(*loss_history)
    plt.plot(steps, losses)
    plt.xlabel("training step (one batch)")
    plt.ylabel("loss")
    plt.title("Training loss (lower is better)")
    plt.show()
else:
    print("No training history yet - run the training cell first.")

### Load saved weights (optional)

Only needed if you skipped training this session. It loads `CHECKPOINT_PATH` into the model.
The data steps above must use the same settings as the run that made the file, so the vocabulary matches.
(On Kaggle, add the output of an earlier run with *Add Input* and point `CHECKPOINT_PATH` at it.)

In [ ]:
saved = torch.load(CHECKPOINT_PATH, map_location="cpu")
if isinstance(saved, dict):
    if saved["used"] != used:
        raise ValueError("This checkpoint was trained with a different vocabulary. Use the data settings it was trained with.")
    saved = saved["params"]
# (Older checkpoints were a plain list of tensors in the same order, so they load too.)

if len(saved) != len(params) or any(p.shape != s.shape for p, s in zip(params, saved)):
    raise ValueError("This checkpoint doesn't fit the model. Check DIMS, N_LAYERS and N_HEADS.")
with torch.no_grad():
    for p, s in zip(params, saved):
        p.copy_(s)
print("loaded", CHECKPOINT_PATH)

## 6. How well did it learn?

Compare the loss on training data with the loss on validation data (text the model never trained on).
If the two are close, the model learned general patterns. If the training loss is much lower, it is starting to memorize.

To picture what a loss number means, raise e (about 2.718) to that power. The result is roughly how many tokens the model is torn between at each step, as if it had narrowed the next token down to that many equally likely choices.
For example, a loss of 4.35 means about e^4.35 ≈ 77 choices. Random guessing would be torn between the whole vocabulary.

For reference, an earlier run with a smaller model (2 blocks, 128 numbers per token) reached a training loss of 4.14 and a validation loss of 4.35.

In [ ]:
train_loss = evaluate(train_x[:len(val_x)], train_y[:len(val_x)])   # a sample the same size as the validation set
val_loss = evaluate(val_x, val_y)
print(f"training loss:   {train_loss:.4f}")
print(f"validation loss: {val_loss:.4f}  (torn between about {math.exp(val_loss):,.0f} tokens, out of {vocab_size:,})")

## 7. Let it write

To write, the model repeatedly scores every token, picks one, adds it to the text, and goes again. It stops when it picks the end marker or reaches `max_new_tokens`.

Two settings control the picking:
- **top_k**: only the `top_k` highest-scoring tokens are allowed. This stops it from choosing something silly.
- **temperature**: below 1 makes it play it safe and stick to likely tokens; above 1 makes it more adventurous (and more random).

Your prompt is lowercased automatically, because the model only ever saw lowercase text.

In [ ]:
@torch.no_grad()
def generate(prompt, max_new_tokens=200, temperature=0.75, top_k=40):
    # The model only saw lowercase text. A trailing space would also become its own rare token.
    prompt = prompt.lower().rstrip()
    gpt2_ids = enc.encode_ordinary(prompt)
    missing = [enc.decode([t]) for t in gpt2_ids if t not in g2l]
    if missing:
        print("note: skipping pieces that never appeared in the training data:", missing)
    ids = [BOUNDARY] + [g2l[t] for t in gpt2_ids if t in g2l]

    for _ in range(max_new_tokens):
        context = torch.tensor([ids[-block_size:]], device=device)
        scores = forward(context)[0, -1]                  # scores for the next token
        cutoff = torch.topk(scores, top_k).values[-1]
        scores[scores < cutoff] = float("-inf")           # keep only the top_k tokens
        probs = (scores / temperature).softmax(dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()
        if next_id == BOUNDARY:                           # the model chose to end the text
            break
        ids.append(next_id)

    return enc.decode([used[i - 1] for i in ids[1:] if i != UNK])

In [ ]:
prompts = [
    "the capital of france is",
    "the color of the ocean is",
    "the river runs through the",
    "to be or not to be, that is the",
]
for p in prompts:
    print(generate(p))
    print("-" * 80)

Try your own:

In [ ]:
print(generate("the meaning of life is", temperature=0.75))

## Ideas to try next

- **More data:** raise `MAX_LINES`, or train for more `EPOCHS` while watching that the validation loss keeps dropping.
- **A different size:** change `DIMS`, `N_LAYERS` or `N_HEADS` and compare the losses.
- **A learning-rate schedule:** start with bigger nudges and shrink them as training goes on.
- **Dropout:** during training, randomly zero out some numbers inside the model so it can't rely on any one of them. This fights memorizing.
- **Weight tying:** reuse the token embedding table as the final output layer. Both are a table with one row per token, and sharing them saves about a third of the parameters.